In [1]:
!pip install -q sentence-transformers

import os
import torch

print("GPU Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device Name:", torch.cuda.get_device_name(0))

print("\nFiles ready in session:")
print([f for f in os.listdir(".") if f.endswith(".csv")])

GPU Available: True
Device Name: Tesla T4

Files ready in session:
['DataSet_cleaned.csv', 'split_indices.csv', 'DataSet.csv']


In [3]:
import pandas as pd
from sentence_transformers import SentenceTransformer

# 1. Load data with Python engine to handle multi-line quotes cleanly
print("Loading CSV files...")
try:
    df = pd.read_csv("DataSet_cleaned.csv", engine="python", on_bad_lines="skip")
except Exception:
    # Fallback to the original uncleaned file if cleaned upload had missing bytes
    df = pd.read_csv("DataSet.csv", engine="python", on_bad_lines="skip")

split_df = pd.read_csv("split_indices.csv")

# Match lengths safely
if len(df) != len(split_df):
    common_len = min(len(df), len(split_df))
    df = df.iloc[:common_len].copy()
    split_df = split_df.iloc[:common_len].copy()

df["split"] = split_df["split"].values

# 2. Combine textual features into unified job descriptions
df["full_text"] = (
    "Title: " + df["title"].fillna("").astype(str) + "\n" +
    "Company: " + df["company_profile"].fillna("").astype(str) + "\n" +
    "Description: " + df["description"].fillna("").astype(str) + "\n" +
    "Requirements: " + df["requirements"].fillna("").astype(str)
)

# 3. Clean target labels (0 = genuine, 1 = fraudulent)
label_map = {'f': 0, 't': 1, 'False': 0, 'True': 1, '0': 0, '1': 1, 0: 0, 1: 1, False: 0, True: 1}
df["target"] = df["fraudulent"].map(label_map).fillna(0).astype(int)

train_df = df[df["split"] == "train"].copy()
test_df = df[df["split"] == "test"].copy()

print(f"Loaded {len(train_df):,} training listings and {len(test_df):,} test listings.")

# 4. Load transformer onto the T4 GPU
print("Loading semantic model onto GPU...")
embedder = SentenceTransformer('all-MiniLM-L6-v2', device='cuda')

# 5. Encode all jobs into semantic vectors
print("Encoding training data on GPU...")
X_train = embedder.encode(train_df["full_text"].tolist(), batch_size=128, show_progress_bar=True)
y_train = train_df["target"].values

print("Encoding test data on GPU...")
X_test = embedder.encode(test_df["full_text"].tolist(), batch_size=128, show_progress_bar=True)
y_test = test_df["target"].values

print(f"✅ Encoding complete! Feature matrix shape: {X_train.shape}")

Loading CSV files...
Loaded 6,450 training listings and 1,441 test listings.
Loading semantic model onto GPU...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

Encoding training data on GPU...


Batches:   0%|          | 0/51 [00:00<?, ?it/s]

Encoding test data on GPU...


Batches:   0%|          | 0/12 [00:00<?, ?it/s]

✅ Encoding complete! Feature matrix shape: (6450, 384)


In [4]:
import joblib
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score

# 1. Train classifier on the semantic vectors (balanced class weight handles the imbalance)
print("Training classifier on semantic vectors...")
clf = LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42)
clf.fit(X_train, y_train)

# 2. Evaluate performance on the test set
test_probs = clf.predict_proba(X_test)[:, 1]
test_preds = (test_probs >= 0.50).astype(int)

print("\n=== TEST SET EVALUATION ===")
print(f"ROC-AUC Score: {roc_auc_score(y_test, test_probs):.4f}\n")
print(classification_report(y_test, test_preds, target_names=["Genuine", "Fraudulent"]))

# 3. Save the model bundle to Colab
bundle = {
    "embedder_name": "all-MiniLM-L6-v2",
    "classifier": clf,
    "decision_threshold": 0.50
}

output_filename = "cybershield_job_detector_v3.pkl"
joblib.dump(bundle, output_filename)
print(f"✅ Model successfully exported to '{output_filename}'")

Training classifier on semantic vectors...

=== TEST SET EVALUATION ===
ROC-AUC Score: 0.8617

              precision    recall  f1-score   support

     Genuine       0.99      0.86      0.92      1383
  Fraudulent       0.17      0.69      0.27        58

    accuracy                           0.85      1441
   macro avg       0.58      0.77      0.60      1441
weighted avg       0.95      0.85      0.89      1441

✅ Model successfully exported to 'cybershield_job_detector_v3.pkl'


In [6]:
# 1. Update scanning function with the calibrated 0.40 threshold
OPTIMAL_THRESHOLD = 0.40

def scan_posting(text: str) -> dict:
    vector = embedder.encode([text.strip()], device='cuda')
    risk_score = float(clf.predict_proba(vector)[0, 1])
    is_fraud = risk_score >= OPTIMAL_THRESHOLD
    return {
        "verdict": "🚨 FRAUD DETECTED" if is_fraud else "✅ GENUINE",
        "risk_percentage": round(risk_score * 100, 2)
    }

print("=== RETEST WITH CALIBRATED THRESHOLD (0.40) ===")
print("Subtle Scam:", scan_posting(subtle_scam))
print("Genuine Job:", scan_posting(genuine_job))

# 2. Re-save bundle with calibrated threshold
bundle = {
    "embedder_name": "all-MiniLM-L6-v2",
    "classifier": clf,
    "decision_threshold": OPTIMAL_THRESHOLD
}
joblib.dump(bundle, "cybershield_job_detector_v3.pkl")
print("\n✅ Calibrated bundle saved as 'cybershield_job_detector_v3.pkl'")

=== RETEST WITH CALIBRATED THRESHOLD (0.40) ===
Subtle Scam: {'verdict': '🚨 FRAUD DETECTED', 'risk_percentage': 47.65}
Genuine Job: {'verdict': '✅ GENUINE', 'risk_percentage': 4.76}

✅ Calibrated bundle saved as 'cybershield_job_detector_v3.pkl'
